In [ ]:
from ipynb.fs.full. coupling_layer_cls import *
import torch.distributions as dist

In [ ]:
# --------- Invertible 1x1 Convolution Layer ---------
class Invertible1x1Conv(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        w_init, _ = torch.linalg.qr(torch.randn(num_channels, num_channels))
        self.weight = nn.Parameter(w_init)

    def forward(self, x, reverse=False):
        B, C, H, W = x.shape
        W_mat = self.weight

        if not reverse:
            W_reshape = W_mat.view(C, C, 1, 1)
            z = F.conv2d(x, W_reshape)
            log_det = H * W * torch.slogdet(W_mat)[1]
            return z, log_det.expand(B)
        else:
            W_inv = torch.inverse(W_mat).view(C, C, 1, 1)
            x = F.conv2d(x, W_inv)
            return x

class ActNormLayer(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        # Register `initialized` as a persistent buffer
        self.register_buffer('initialized', torch.tensor(False))  
        self.scale = nn.Parameter(torch.ones(1, num_channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, num_channels, 1, 1))

    def initialize_parameters(self, x):
        with torch.no_grad():
            mean = x.mean(dim=[0, 2, 3], keepdim=True)
            std = x.std(dim=[0, 2, 3], keepdim=True)
            self.bias.data.copy_(-mean)
            self.scale.data.copy_(1 / (std + 1e-6))
            self.initialized.fill_(True)  


    def forward(self, x, reverse=False):
        if not self.initialized.item():  
            self.initialize_parameters(x)

        if reverse:
            x = (x - self.bias) / self.scale
            log_det = -torch.sum(torch.log(torch.abs(self.scale))) * x.size(2) * x.size(3)
        else:
            x = self.scale * x + self.bias
            log_det = torch.sum(torch.log(torch.abs(self.scale))) * x.size(2) * x.size(3)

        log_det = log_det.expand(x.size(0)) 
        return x, log_det



class Squeeze(nn.Module):
    def forward(self, x):
        B, C, H, W = x.size()
        assert H % 2 == 0 and W % 2 == 0
        x = x.view(B, C, H // 2, 2, W // 2, 2)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        return x.view(B, C * 4, H // 2, W // 2)


class Unsqueeze(nn.Module):
    def forward(self, x):
        B, C, H, W = x.size()
        assert C % 4 == 0
        x = x.view(B, C // 4, 2, 2, H, W)
        x = x.permute(0, 1, 4, 2, 5, 3).contiguous()
        return x.view(B, C // 4, H * 2, W * 2)

class SplitLayer(nn.Module):
    def forward(self, x):
        C = x.size(1)
        return x[:, :C // 2], x[:, C // 2:]

class MergeLayer(nn.Module):
    def forward(self, x1, x2):
        return torch.cat([x1, x2], dim=1)


class CondMSRealNVP(nn.Module):
    def __init__(self, num_coupling_layers, num_classes, input_shape=(1, 182, 182), 
                 squeeze_layers=None, split_layers=None, device=None, 
                 layer_filters=None, mask_types=None, verbose=False):
        super(CondMSRealNVP, self).__init__()
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.input_shape = input_shape
        self.verbose = verbose
        self.num_coupling_layers = num_coupling_layers
        self.num_classes = num_classes
        self.squeeze_layers = squeeze_layers or []
        self.split_layers = split_layers or []
        
        assert layer_filters is not None and len(layer_filters) == num_coupling_layers, \
            "You must provide a list of filter tuples, one per coupling layer."
        assert mask_types is not None and len(mask_types) == num_coupling_layers, \
            "You must provide a list of mask types, one per coupling layer."

        self.mask_types = mask_types

        self.squeeze = Squeeze()
        self.unsqueeze = Unsqueeze()
        self.split = SplitLayer()
        self.merge = MergeLayer()

        self.saved_shapes = {}
        self.saved_parts = {}

        current_shape = input_shape
        self.shapes = []
        self.layer_channels = []
        
        for i in range(num_coupling_layers):
            if i in self.squeeze_layers:
                current_shape = (
                    current_shape[0] * 4,
                    current_shape[1] // 2,
                    current_shape[2] // 2
                )
            self.shapes.append(current_shape)
            self.layer_channels.append(current_shape[0])

            if i in self.split_layers:
                split_shape = (
                    current_shape[0] // 2,
                    current_shape[1],
                    current_shape[2]
                )
                self.saved_shapes[i] = split_shape
                current_shape = split_shape
                
        latent_dim = current_shape[0] * current_shape[1] * current_shape[2]
        for s in self.saved_shapes.values():
            latent_dim += s[0] * s[1] * s[2]

        self.distribution = dist.MultivariateNormal(
            torch.zeros(latent_dim, device=self.device),
            torch.eye(latent_dim, device=self.device)
        )

        self.coupling_layers = nn.ModuleList([
            Coupling(input_shape=self.shapes[i], num_classes=num_classes, 
                     filters=layer_filters[i], verbose=verbose).to(self.device)
            for i in range(num_coupling_layers)
        ])
        self.inv_convs = nn.ModuleList([
            Invertible1x1Conv(self.layer_channels[i]).to(self.device)
            for i in range(num_coupling_layers)
        ])
        self.actnorms = nn.ModuleList([
            ActNormLayer(self.layer_channels[i]).to(self.device)
            for i in range(num_coupling_layers)
        ])

    def create_mask(self, layer_idx, x):
        B, C, H, W = x.shape
        mask_type = self.mask_types[layer_idx]
        
        if mask_type == 'custom':
            if self.custom_mask_tensor is None:
                raise ValueError("Custom mask selected but no custom_mask_tensor is defined.")
        
            mask = self.custom_mask_tensor.to(x.device)
            if mask.ndim == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
                mask = mask.expand(B, C, H, W)
            elif mask.ndim == 4:
                mask = mask.expand(B, C, H, W)
            else:
                raise ValueError("custom_mask_tensor must be shape (H, W) or (B, C, H, W)")
                
            # Invert mask if layer index is odd
            if layer_idx % 2 == 1:
                mask = 1 - mask
                
            return mask

        elif mask_type == 'checkerboard':
            base_mask = (torch.arange(H, device=x.device).unsqueeze(1) + torch.arange(W, device=x.device).unsqueeze(0)) % 2
            base_mask = base_mask.float()
            masks = [(base_mask if (layer_idx + c) % 2 == 0 else 1 - base_mask) for c in range(C)]
            return torch.stack(masks, dim=0).unsqueeze(0)  # (1, C, H, W)

        elif mask_type == 'channel':
            half = C // 2
            if layer_idx % 2 == 0:
                mask = torch.cat([
                    torch.ones(half, device=x.device),
                    torch.zeros(C - half, device=x.device)
                ])
            else:
                mask = torch.cat([
                    torch.zeros(half, device=x.device),
                    torch.ones(C - half, device=x.device)
                ])
            mask = mask.view(1, C, 1, 1).expand(B, -1, H, W)
            return mask
        
        elif mask_type == 'vertical_stripe':
            # Alternate columns: 1, 0, 1, 0, ...
            base_mask = (torch.arange(W, device=x.device) % 2).float()
            base_mask = base_mask.unsqueeze(0).expand(H, -1)  # shape (H, W)
            masks = [(base_mask if (layer_idx + c) % 2 == 0 else 1 - base_mask) for c in range(C)]
            return torch.stack(masks, dim=0).unsqueeze(0)  # (1, C, H, W)

        elif mask_type == 'horizontal_stripe':
            # Alternate rows: 1, 0, 1, 0, ...
            base_mask = (torch.arange(H, device=x.device) % 2).float()
            base_mask = base_mask.unsqueeze(1).expand(-1, W)  # shape (H, W)
            masks = [(base_mask if (layer_idx + c) % 2 == 0 else 1 - base_mask) for c in range(C)]
            return torch.stack(masks, dim=0).unsqueeze(0)  # (1, C, H, W)
        
        elif mask_type == 'block_checkerboard':
            block_size = self.block_checkerboard_size
            grid_H = (H + block_size - 1) // block_size
            grid_W = (W + block_size - 1) // block_size

            base_pattern = (torch.arange(grid_H, device=x.device).unsqueeze(1) +
                            torch.arange(grid_W, device=x.device).unsqueeze(0)) % 2
            base_pattern = base_pattern.float()
            mask = base_pattern.repeat_interleave(block_size, dim=0).repeat_interleave(block_size, dim=1)
            mask = mask[:H, :W]  

            masks = [(mask if (layer_idx + c) % 2 == 0 else 1 - mask) for c in range(C)]
            return torch.stack(masks, dim=0).unsqueeze(0)  # (1, C, H, W)

        else:
            raise ValueError(f"Unknown mask type '{mask_type}' at layer {layer_idx}")

    def forward(self, x, y, reverse=False, verbose=None, return_z_list = False):
        if verbose is None:
            verbose = self.verbose
        log_det_inv = 0

        if not reverse:
            self.saved_parts = {}
            for i in range(self.num_coupling_layers):
                if verbose:
                    print(f"\n=== Forward Layer {i} ===")
                    print(f"Input shape: {x.shape}")
                    
                if i in self.squeeze_layers:
                    x = self.squeeze(x)
                    if verbose: print(f"[Layer {i}] Squeeze → {x.shape}")
                        
                x, log_det = self.flow_step(x, y, i, reverse=False, verbose=verbose)
                log_det_inv += log_det
                
                if i in self.split_layers:
                    x, saved = self.split(x)
                    self.saved_parts[i] = saved
                    if verbose: print(f"[Layer {i}] Split → main: {x.shape}, saved: {saved.shape}")
                        
            if return_z_list:
                 z_list = [x] + [self.saved_parts[idx] for idx in sorted(self.saved_parts.keys())]
            
            z = [x.view(x.shape[0], -1)]
            for idx in sorted(self.saved_parts.keys()):
                z.append(self.saved_parts[idx].view(x.shape[0], -1))
            z = torch.cat(z, dim=1)

            if verbose:
                print(f"\n=== Forward Complete ===")
                print(f"Final latent shape: {z.shape}")
            if return_z_list == True:
                return z, log_det_inv, z_list
            else:
                return z, log_det_inv

        else:
            batch_size = x.shape[0]
            total_dims = x.shape[1]
            
            flat_sizes = []
            final_shape = self.shapes[-1]
            flat_sizes.append(final_shape[0] * final_shape[1] * final_shape[2])
            for shape in self.saved_shapes.values():
                flat_sizes.append(shape[0] * shape[1] * shape[2])

            assert sum(flat_sizes) == total_dims
            
            splits = torch.split(x, flat_sizes, dim=1)
            x_final = splits[0].view(batch_size, *final_shape)

            saved_parts = {}
            for idx, shape in zip(sorted(self.saved_shapes.keys()), self.saved_shapes.values()):
                saved_parts[idx] = splits[len(saved_parts)+1].view(batch_size, *shape)

            for i in reversed(range(self.num_coupling_layers)):
                if verbose:
                    print(f"\n=== Reverse Layer {i} ===")
                    print(f"Input shape: {x_final.shape}")

                if i in self.split_layers:
                    x_final = self.merge(x_final, saved_parts[i])
                    if verbose: print(f"[Rev Layer {i}] Merge ← {x_final.shape}")

                x_final, _ = self.flow_step(x_final, y, i, reverse=True, verbose=verbose)

                if i in self.squeeze_layers:
                    x_final = self.unsqueeze(x_final)
                    if verbose: print(f"[Rev Layer {i}] Unsqueeze ← {x_final.shape}")

            if verbose:
                print(f"\n=== Reverse Complete ===")
                print(f"Reconstructed input shape: {x_final.shape}")
            return x_final

    def flow_step(self, x, y, step_idx, reverse=False, verbose=None):
        if verbose is None:
            verbose = self.verbose
        log_det = 0
        B, C, H, W = x.shape
        mask = self.create_mask(step_idx, x).expand(B, -1, -1, -1)
        reversed_mask = 1 - mask
        if verbose:
            mask_type = self.mask_types[step_idx]
            print(f"  [Mask Info] Type: {mask_type}")
            print(f"  [Mask Values] Channel 0 (3x3): {mask[0, 0, :3, :3]}")
            second_idx = mask.shape[1] // 2
            if mask.shape[1] > 1:
                print(f"  [Mask Values] Channel {second_idx} (3x3):\n{mask[0, second_idx, :3, :3]}")

        if not reverse:
            if verbose: print(f"\n-- Flow Step {step_idx} FORWARD --")

            x, ld = self.actnorms[step_idx](x, reverse=False)
            if verbose: print(f"[ActNorm] -> {x.shape}")
            log_det += ld

            x, ld = self.inv_convs[step_idx](x, reverse=False)
            if verbose: print(f"[InvConv] -> {x.shape}")
            log_det += ld

            x_masked = x * mask
            s1, t1 = self.coupling_layers[step_idx](x_masked, y)
            s1, t1 = s1 * reversed_mask, t1 * reversed_mask
            x = reversed_mask * (x * torch.exp(s1) + t1) + x_masked
            log_det += torch.sum(s1, dim=[1, 2, 3])

            x_masked = x * reversed_mask
            s2, t2 = self.coupling_layers[step_idx](x_masked, y)
            s2, t2 = s2 * mask, t2 * mask
            x = mask * (x * torch.exp(s2) + t2) + x_masked
            if verbose: print(f"[Coupling] Output -> {x.shape}")
            log_det += torch.sum(s2, dim=[1, 2, 3])

            return x, log_det

        else:
            if verbose: print(f"\n-- Flow Step {step_idx} REVERSE --")

            x_masked = x * reversed_mask
            s2, t2 = self.coupling_layers[step_idx](x_masked, y)
            s2, t2 = s2 * mask, t2 * mask
            x = mask * ((x - t2) * torch.exp(-s2)) + x_masked
            if verbose: print(f"[Rev Coupling 2] Output <- {x.shape}")

            x_masked = x * mask
            s1, t1 = self.coupling_layers[step_idx](x_masked, y)
            s1, t1 = s1 * reversed_mask, t1 * reversed_mask
            x = reversed_mask * ((x - t1) * torch.exp(-s1)) + x_masked
            if verbose: print(f"[Rev Coupling 1] Output <- {x.shape}")

            x = self.inv_convs[step_idx](x, reverse=True)
            if verbose: print(f"[Rev InvConv] -> {x.shape}")

            x, _ = self.actnorms[step_idx](x, reverse=True)
            if verbose: print(f"[Rev ActNorm] -> {x.shape}")

            return x, 0

    def compute_loss(self, x, y, verbose=None):
        if verbose is None:
            verbose = self.verbose 
        x, y = x.to(self.device), y.to(self.device)
        z_pred, log_det_inv = self(x, y, verbose=verbose)
        log_likelihood_z = self.distribution.log_prob(z_pred)
        return torch.mean(log_likelihood_z), torch.mean(log_det_inv)

    def train_step(self, x, y, optimizer, verbose=None):
        if verbose is None:
            verbose = self.verbose
        optimizer.zero_grad()
        log_likelihood_z, log_det_inv = self.compute_loss(x, y, verbose=verbose)
        loss = -(log_likelihood_z + log_det_inv)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
        optimizer.step()
        return loss.item(), log_likelihood_z.item(), log_det_inv.item()

    def predict(self, x, y, verbose=None):
        if verbose is None:
            verbose = self.verbose
        z_pred, log_det_inv = self(x, y, reverse=False, verbose=verbose)
        log_likelihood_z = self.distribution.log_prob(z_pred)
        return log_likelihood_z + log_det_inv

    def predict_for_all_labels(self, x, verbose=None):
        if verbose is None:
            verbose = self.verbose
        all_log_likelihoods = []
        for i in range(self.num_classes):
            y = torch.zeros((x.shape[0], self.num_classes), device=x.device)
            y[:, i] = 1
            log_likelihood = self.predict(x, y, verbose=verbose)
            all_log_likelihoods.append(log_likelihood)
        return torch.stack(all_log_likelihoods, dim=1)
    
    def extract_log_det_maps(self, x, y):
        """
        Extracts the `s1` and `s2` maps (log-scale factors) for each coupling step,
        grouping them by shape `(C, H, W)`.
        
        Returns:
            dict: keys = (C, H, W), values = list of tensors with shape (B, C, H, W)
        """
        log_det_maps = {}
        
        with torch.no_grad():
            for i in range(self.num_coupling_layers):
                if i in self.squeeze_layers:
                    x = self.squeeze(x)
    
                B, C, H, W = x.shape
                key = (C, H, W)
                if key not in log_det_maps:
                    log_det_maps[key] = []
    
                # --- Masks ---
                mask = self.create_mask(i, x)
                reversed_mask = 1 - mask
    
                # Coupling 1
                x_masked = x * mask
                s1, t1 = self.coupling_layers[i](x_masked, y)
                s1 = s1 * reversed_mask
                x = reversed_mask * (x * torch.exp(s1) + t1) + x_masked
                log_det_maps[key].append(s1.clone())
    
                # Coupling 2
                x_masked = x * reversed_mask
                s2, t2 = self.coupling_layers[i](x_masked, y)
                s2 = s2 * mask
                x = mask * (x * torch.exp(s2) + t2) + x_masked
                log_det_maps[key].append(s2.clone())
    
                if i in self.split_layers:
                    x, _ = self.split(x)
    
        return log_det_maps